# CS2 教练小模型 · Colab 训练（T4 16GB）

把 Mac 上跑不动的 A 层 SFT 搬到 Colab。对应本地 `learn/train_sft.py`（MLX 版）的 CUDA 孪生版。

**三步**：装依赖 → 传数据 → 开训 → 下载适配器。按顺序点 ▶ 即可。

- 底座：Qwen2.5-1.5B-Instruct（和本地 `model.yaml` 同源，自动映射）
- 显存：4bit QLoRA，T4 16G 约占 6–8GB，够用
- 数据：本地 `build_sft.py` 产出的 `train.jsonl / valid.jsonl`（chat 或 completion 格式都吃）

## 0) 确认 GPU（必须是 T4/L4/A100 其中之一）

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

如果上面报错：菜单 **代码执行程序 → 更改运行时类型 → T4 GPU**，再重跑。CPU 硬跑会被脚本拒绝。

## 1) 装依赖（约 3–5 分钟）

In [ ]:
!pip install -q bitsandbytes accelerate peft transformers datasets pyyaml huggingface_hub

## 2) 拿代码（二选一）

**A. 仓库能从 GitHub 拉**（把 URL 换成你的）：

In [ ]:
# !git clone <你的仓库URL> cs2coach && cd cs2coach && git pull
!ls

**B. 直接上传两个文件**（推荐：本地跑不动 git 就用这个）

把本地这三个文件传上来：`learn/colab/train_sft_colab.py`、`learn/build_sft.py`、`learn/configs/model.yaml`。

In [ ]:
from google.colab import files
up = files.upload()  # 选中 train_sft_colab.py / build_sft.py / model.yaml
!ls -la

## 3) 传 SFT 数据

本地先生成（Mac/任何机器都行，只要标准库）：

```bash
python3 learn/build_sft.py --input out/rounds.jsonl --out-dir learn/data/sft --format chat
```

然后把 `train.jsonl` / `valid.jsonl` 传上来。已有就直接上传，无需重跑上面那行。

In [ ]:
!mkdir -p sft && ls sft/ 2>/dev/null || echo "(空：下面上传)"
from google.colab import files
up2 = files.upload()  # 选中 train.jsonl / valid.jsonl
!mkdir -p sft && mv -f train.jsonl valid.jsonl sft/ 2>/dev/null; ls -la sft/

## 4) 审计（不训练，30 秒）

In [ ]:
!python3 train_sft_colab.py --check --data-dir sft 2>&1 | head -n 40
# 如果脚本在仓库里：!python3 learn/colab/train_sft_colab.py --check --data-dir sft

上面出现 `!` 的行先修：缺数据就回第 3 步，缺依赖就回第 1 步。

## 5) 小跑验证（50 步，约 10 分钟）

先确认 loss 在降、显存不爆，再全量。

In [ ]:
!python3 train_sft_colab.py --data-dir sft --out-dir coach-adapter-colab --max-steps 50

## 6) 全量（默认 600 步，和本地 model.yaml 一致，约 1–2 小时）

In [ ]:
!python3 train_sft_colab.py --data-dir sft --out-dir coach-adapter-colab

## 7) 抽查一句话（适配器真学会说话了没）

In [ ]:
!python3 train_sft_colab.py --data-dir sft --out-dir coach-adapter-colab --test-prompt "第7回合，CT 3v5 回防，道具落后2颗。发生了什么？下把怎么改？" --max-new-tokens 256

## 8) 打包下载（拉回 Mac）

In [ ]:
!zip -qr coach-adapter-colab.zip coach-adapter-colab && ls -lh coach-adapter-colab.zip
from google.colab import files
files.download('coach-adapter-colab.zip')

下载后解压到 `learn/models/coach-adapter-colab/` 即可。

---

### 附：胜率模型（LightGBM，不吃 GPU，Colab 免费 CPU 就能跑）

```
!pip install -q lightgbm scikit-learn pandas pyyaml
!python3 learn/train_winrate.py --check --match-json out/
!python3 learn/train_winrate.py --match-json out/ --out-dir learn/models/winrate
```